In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from ReasoningGraph import ReasoningGraph

In [ ]:
model_name = "Qwen/Qwen3-4B-Thinking-2507"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

In [ ]:
reasoning_graph = ReasoningGraph()

In [ ]:
messages = [
  {
    "content": "Question: Find the domain of the expression $\\frac{\\sqrt{x-2}}{\\sqrt{5-x}}$.}\nAnswer:The expressions inside each square root must be non-negative.\nTherefore, $x-2 \\ge 0$, so $x\\ge2$, and $5 - x \\ge 0$, so $x \\le 5$.\nAlso, the denominator cannot be equal to zero, so $5-x>0$, which gives $x<5$.\nTherefore, the domain of the expression is $\\boxed{[2,5)}$.\n\nQuestion: If $\\det \\mathbf{A} = 2$ and $\\det \\mathbf{B} = 12,$ then find $\\det (\\mathbf{A} \\mathbf{B}).$\nAnswer:We have that $\\det (\\mathbf{A} \\mathbf{B}) = (\\det \\mathbf{A})(\\det \\mathbf{B}) = (2)(12) = \\boxed{24}.$\n\nQuestion: Terrell usually lifts two 20-pound weights 12 times. If he uses two 15-pound weights instead, how many times must Terrell lift them in order to lift the same total weight?\nAnswer:If Terrell lifts two 20-pound weights 12 times, he lifts a total of $2\\cdot 12\\cdot20=480$ pounds of weight.  If he lifts two 15-pound weights instead for $n$ times, he will lift a total of $2\\cdot15\\cdot n=30n$ pounds of weight.  Equating this to 480 pounds, we can solve for $n$: \\begin{align*}\n30n&=480\\\\\n\\Rightarrow\\qquad n&=480/30=\\boxed{16}\n\\end{align*}\n\nQuestion: If the system of equations\n\n\\begin{align*}\n6x-4y&=a,\\\\\n6y-9x &=b.\n\\end{align*}has a solution $(x, y)$ where $x$ and $y$ are both nonzero, find $\\frac{a}{b},$ assuming $b$ is nonzero.\nAnswer:If we multiply the first equation by $-\\frac{3}{2}$, we obtain\n\n$$6y-9x=-\\frac{3}{2}a.$$Since we also know that $6y-9x=b$, we have\n\n$$-\\frac{3}{2}a=b\\Rightarrow\\frac{a}{b}=\\boxed{-\\frac{2}{3}}.$$\n\nQuestion: What is the modulo $13$ residue of $247+5 \\cdot 39 + 7 \\cdot 143 +4 \\cdot 15?$",
    "role": "user"
  }
]
ground_truth = 8

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate using the custom generate code
generated_ids = model.generate(
    input_ids=model_inputs["input_ids"],
    custom_generate=reasoning_graph.reasoning_graph_builder,
    use_cache=True,
    max_new_tokens=2048,
    do_sample=True,
    attention_mask=model_inputs["attention_mask"],
    temperature=1
)


In [ ]:
print(len(reasoning_graph.metrics))

In [ ]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

print("content:", content)

In [ ]:
cur = reasoning_graph.head_node
count = 0
while len(cur.children):
  count += 1
  print(f'{tokenizer.decode(cur.token_val, skip_special_tokens=True)} -> {cur.children_dists[0]}')
  print()
  cur = cur.children[0]


print(f'Total: {count}')


In [ ]:
sorted_metrics = sorted(reasoning_graph.metrics, reverse=True)
wanted_idx = int(0.2 * len(sorted_metrics)) - 1

# Preventing 0 entropy at cutoff
while sorted_metrics[wanted_idx] <= 0:
    wanted_idx -= 1

val = sorted_metrics[wanted_idx]
print(wanted_idx)